In [ ]:
# GPU 런타임으로 전환
# 상단 메뉴에서 런타임 -> 런타임 유형 변경 선택
# 하드웨어 가속기에서 T4 GPU 선택하고 저장


# 해당 코드는 구글 코랩에서 실행하는것으로 구성되어있습니다.

In [8]:
# gpu 확인
!nvidia-smi

/bin/bash: 줄 1: nvidia-smi: 명령어를 찾을 수 없음


In [ ]:
# 런타임 환경 바꾸고 라이브러리 설치해줍니다
%pip install ultralytics

In [ ]:
# 압축 해제
# 경로 맞춰놨으나 오류가 날 경우 경로 꼭 확인!
!unzip -q /content/dataset-obb.zip -d /content/dataset-obb

In [ ]:
# <<개선 시도 모델>>
# 모델 개선을 위해 사용해봤으나 <<최종 기본 모델>> 더 나았음
# 모델 학습
# yolo11m-obb.pt (중간 크기로 메모리 요구가 yolo11n-obb 보다 큼)
# 데이터 증강 사용 (회전 허용 (±10도), 스케일 변환, 이동 허용, 좌우 반전 확률 50%, 모자이크 합성 확률 70%, 복붙 증강,	자동 증강)
# epochs - 전체 학습 데이터셋을 n회 반복 학습.
# imgsz - 입력 이미지 한 변 크기(정사각형). (OBB는 일반적으로 640/1024 사용)
# batch - GPU 한 번 전송 이미지 수
# degrees - 회전 증강 관련 각도 (증강 시 10° 범위 내 회전)
# scale - 이미지 및 라벨을 최대 50 % 확대 및 축소
# translate - 이미지 및 라벨을 가로·세로 최대 10 % 만큼 무작위로 평행 이동
# fliplr - 절반은 랜덤으로 좌우가 반전 나머지 절만은 원본 방향 유지
# mosaic - 4장의 이미지를 무작위로 잘라 2×2 격자로 붙여 한 장으로 만드는 증강
# copy_paste - 빈도가 낮은 클래스를 인위적으로 늘려 검출률을 개선
# auto_augment - 조명·색상 변화까지 학습해 실전 노이즈에 대비
# close_mosaic - 학습 후반에 mosaic 증강 끔
# cos_lr - 학습 속도 관련 코사인처럼 부드럽게 학습하여 실수 줄인다.
!yolo obb train \
  model=yolo11m-obb.pt \
  data=/content/dataset-obb/dataset-obb/obb/obb.yaml \
  epochs=100 imgsz=640 batch=16 \
  degrees=10 scale=0.5 translate=0.1 fliplr=0.5 \
  mosaic=0.7 copy_paste=0.1 auto_augment=randaugment \
  close_mosaic=10 cos_lr=True

In [1]:
# <<최종 기본 모델>> - floor_plan에서 사용 중인 모델
# 해당 셀 실행
# 모델 학습
# yolo11n-obb.pt라는 경량 모델을 사용
# 4가지 인자(data, model, epochs, imgsz=640) 외 기본 설정만으로 학습
#!yolo obb train data=/home/zen35/Desktop/musoft/dataset/obb.yaml model=yolo11n-obb.pt epochs=100 imgsz=640

!yolo obb train data=/home/zen35/Desktop/imagedataset/test/obb.yaml model=yolo11n-obb.pt epochs=100 imgsz=640



Ultralytics 8.3.168 🚀 Python-3.10.18 torch-2.7.1+cu126 CPU (Intel Core(TM) i3-10105 3.70GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/zen35/Desktop/imagedataset/test/obb.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, 

In [ ]:
# 캐시 삭제 - 모델 학습 시키고 나서 다시 학습 시키게 될 때 해야 함
!rm -rf /content/dataset-obb/obb/labels/train.cache
!rm -rf /content/dataset-obb/obb/labels/val.cache

In [1]:
# 학습된 모델 테스트 및 시각화 - root 위치에서 test_images 폴더 생성 후 이미지 업로드

import os
from glob import glob
from ultralytics import YOLO
from IPython.display import Image, display

# 모델 로드
model = YOLO("/home/zen35/Documents/best9.pt")

# 이미지 폴더 경로
image_folder = "/home/zen35/Desktop/musoft"  # 이미지가 들어 있는 폴더
image_paths = sorted(glob(os.path.join(image_folder, "*.jpg")))

# 예측 결과 저장 디렉토리 (자동으로 runs/obb/predictN 폴더 생성됨)
results = model.predict(source=image_folder, save=True, save_json=True, conf=0.3, iou=0.3)

# 결과 이미지가 저장된 폴더 찾기
latest_predict_dir = sorted(glob("runs/obb/predict*"))[-1]

# 결과 이미지 경로 리스트
predicted_images = sorted(glob(os.path.join(latest_predict_dir, "*.jpg")))

# 결과 출력
for pred_img in predicted_images:
    print(f"Showing: {os.path.basename(pred_img)}")
    display(Image(filename=pred_img))



image 1/110 /home/zen35/Desktop/musoft/1_0.png: 640x544 None80.0ms
image 2/110 /home/zen35/Desktop/musoft/1_1.png: 640x544 None57.5ms
image 3/110 /home/zen35/Desktop/musoft/1_10.png: 640x544 None64.4ms
image 4/110 /home/zen35/Desktop/musoft/1_11.png: 640x544 None57.2ms
image 5/110 /home/zen35/Desktop/musoft/1_12.png: 640x544 None61.1ms
image 6/110 /home/zen35/Desktop/musoft/1_13.png: 640x544 None62.7ms
image 7/110 /home/zen35/Desktop/musoft/1_14.png: 640x544 None67.7ms
image 8/110 /home/zen35/Desktop/musoft/1_15.png: 640x544 None59.1ms
image 9/110 /home/zen35/Desktop/musoft/1_16.png: 640x544 None56.2ms
image 10/110 /home/zen35/Desktop/musoft/1_17.png: 640x544 None75.6ms
image 11/110 /home/zen35/Desktop/musoft/1_18.png: 640x544 None59.6ms
image 12/110 /home/zen35/Desktop/musoft/1_19.png: 640x544 None60.1ms
image 13/110 /home/zen35/Desktop/musoft/1_2.png: 640x544 None69.1ms
image 14/110 /home/zen35/Desktop/musoft/1_3.png: 640x544 None60.1ms
image 15/110 /home/zen35/Desktop/musoft/1_4.pn

IndexError: list index out of range

In [2]:
# 만들어진 모델 다운로드

from google.colab import files
files.download('runs/obb/train/weights/best.pt')  # 파일의 실제 경로 다시 한 번 확인

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# (단순 확인용) 이미지 잘 매칭되어있는지 확인용(라벨-이미지)
import os

image_dir = '/content/dataset-obb/dataset-obb/obb/images/val'
label_dir = '/content/dataset-obb/dataset-obb/obb/labels/val'

image_exts = ['.jpg', '.jpeg', '.png']
label_ext = '.txt'

# 이미지 이름 리스트 (확장자 제거)
image_names = [os.path.splitext(f)[0] for f in os.listdir(image_dir) if os.path.splitext(f)[1].lower() in image_exts]
label_names = [os.path.splitext(f)[0] for f in os.listdir(label_dir) if f.endswith(label_ext)]

image_set = set(image_names)
label_set = set(label_names)

only_images = image_set - label_set
only_labels = label_set - image_set
matched = image_set & label_set

print(f"일치하는 이미지-라벨 쌍 수: {len(matched)}")
print(f"라벨 없이 존재하는 이미지 수: {len(only_images)}")
print(f"이미지 없이 존재하는 라벨 수: {len(only_labels)}")

if only_images:
    print("\n 라벨 없이 존재하는 이미지 파일:")
    for name in sorted(only_images):
        print(f"- {name}")

if only_labels:
    print("\n 이미지 없이 존재하는 라벨 파일:")
    for name in sorted(only_labels):
        print(f"- {name}")
